# Exploring open geospatial data sources

The `geodatacatalog.explorer` sub-project ships a curated registry of ~90 open services
([SOURCES.md](../SOURCES.md)). This notebook walks the full workflow:
**search → check → list layers → preview → generate catalog.yml**.

Everything is also available from the terminal: `geodatacatalog --help`.

In [1]:
import logging
from geodatacatalog.explorer import (
    load_registry, search_sources, check_sources, list_layers,
    preview, to_catalog_provider, merge_into_catalog, get_source,
)

logging.basicConfig(level=logging.INFO, format="%(message)s")
sources = load_registry()
len(sources)

157

## 1. Search by theme — French or English

The synonym table maps both languages to canonical themes
(`forêt` = `forest` = `canopy`, `bathymétrie` = `bathymetry`...).

In [2]:
for src, score in search_sources("forêt", sources, coverage="france")[:5]:
    print(f"{src.id:<22} {src.type:<5} {src.coverage:<8} {src.notes[:60]}")

ign_wfs                wfs   france   BD TOPO, cadastre, RPG, ZNIEFF/Natura 2000, BD Forêt... (hun
ign_wmts               wmts  france   Orthophotos, Plan IGN v2, BD Forêt v2, Corine Land Cover
gee_community          registry global   Catalogue communautaire Earth Engine (~4400 datasets hébergé
sentinelhub            rest  global   THE source for named spectral INDICES — any spyndex Sentinel
gfw_data_api           rest  global   Deforestation and forest change: tree cover loss, GLAD and R


## 2. Check that services are alive

`check_sources` probes each service (GetCapabilities, `/collections`, plain GET)
and records `status` in the registry. The CLI variant also rewrites
`sources.yml` and regenerates `SOURCES.md`.

In [6]:
subset = [s for s, _ in search_sources("soil", sources)]
check_sources(subset, timeout=15)

Checking 7 sources (concurrency=10, timeout=15s)
  ✓ usda_sda                 ok               975 ms  HTTP 200
  ✓ gee_community            ok              1030 ms  HTTP 200
  ✓ openlandmap              ok              1238 ms  HTTP 200
  ✓ soilgrids_rest           ok              1451 ms  HTTP 200
  ✓ soilgrids_clay           ok              2718 ms  GetCapabilities OK
  ✓ soilgrids_soc            ok              2783 ms  GetCapabilities OK
  ✓ soilgrids_sand           ok              3261 ms  GetCapabilities OK


{'ok': 7}

## 3. List the layers of a source

In [4]:
src = get_source(sources, "ign_wfs")
layers = list_layers(src, grep="commune")
layers[:5]

[{'id': 'LIMITES_ADMINISTRATIVES_EXPRESS.LATEST:chef_lieu_de_commune',
  'title': 'ADMIN EXPRESS mises a jour en continu - chef lieu de commune',
  'description': 'ADMIN EXPRESS mises a jour en continu - chef lieu de commune ; édition 2026-08-25',
  'keywords': ['features', 'chef_lieu_de_commune'],
  'bbox': [-63.2812, -21.4215, 55.8984, 51.2711],
  'legend': None,
  'sends_datetime': False},
 {'id': 'LIMITES_ADMINISTRATIVES_EXPRESS.LATEST:chef_lieu_de_commune_associee_ou_deleguee',
  'title': 'ADMIN EXPRESS mises a jour en continu - chef lieu de commune associée ou déléguée',
  'description': 'ADMIN EXPRESS mises a jour en continu - chef lieu de commune associée ou déléguée ; édition 2026-08-25',
  'keywords': ['features', 'chef_lieu_de_commune_associee_ou_deleguee'],
  'bbox': [-63.2812, -21.4215, 55.8984, 51.2711],
  'legend': None,
  'sends_datetime': False},
 {'id': 'LIMITES_ADMINISTRATIVES_EXPRESS.LATEST:commune',
  'title': 'ADMIN EXPRESS mises a jour en continu - commune',
  'd

## 4. Preview a layer on a small bbox

The preview uses the *same* layer classes as `catalog.fetch()`,
so what you see is what the catalog will return.

In [5]:
m = preview(src, "ADMINEXPRESS-COG.LATEST:commune", bbox=(1.40, 43.55, 1.50, 43.65))
m

load_catalog: 1 layers loaded from /var/folders/4d/_1r4_wcj0pb80gfn213s98gc0000gr/T/gdc_build_qgx83513/catalog.yml
[ign_wfs__ADMINEXPRESS_COG_LATEST_commune] WFS pagination: 8 features
[ign_wfs__ADMINEXPRESS_COG_LATEST_commune] WFS: 8 features
[ign_wfs__ADMINEXPRESS_COG_LATEST_commune] OK (downloaded) [WFS] — 8 features
[ign_wfs__ADMINEXPRESS_COG_LATEST_commune] columns:
  ✓ gml_id — ex: commune.11355
  ✓ cleabs — ex: COMMUNE_0000000000031044
  ✓ nom_officiel — ex: Balma
  ✓ nom_officiel_en_majuscules — ex: BALMA
  ✓ statut — ex: Commune simple
  ✓ code_insee — ex: 31044
  ✓ population — ex: 17772
  ✓ date_du_recensement — ex: 2023-01-01
  ✓ organisme_recenseur — ex: INSEE
  ✓ code_insee_du_canton — ex: 3124
  ✓ code_insee_de_l_arrondissement — ex: 313
  ✓ code_insee_du_departement — ex: 31
  ✗ code_insee_de_la_region — ex: 76
  ✗ code_siren — ex: 213100449
  ✗ codes_siren_des_epci — ex: 243100518
  ✗ code_postal — ex: 31130
  ✗ superficie_cadastrale — ex: 1660


## 5. Generate a catalog.yml

`merge_into_catalog` creates the file or merges into an existing one, and
validates the result with `load_catalog()` before writing.

In [ ]:
block = to_catalog_provider(src, ["ADMINEXPRESS-COG.LATEST:commune"])
merge_into_catalog(block, "my_catalog.yml")

from geodatacatalog import load_catalog, GeoRequest
catalog = load_catalog("my_catalog.yml")
results = catalog.fetch(GeoRequest(bbox=(1.40, 43.55, 1.50, 43.65)))
results